# Update Gronings↔Dutch Translation Model

This notebook walks through the full pipeline for updating the
[HuggingFace model](https://huggingface.co/Tom9358/nllb-tatoeba-gos-nld-v1):

1. **Download** the latest Tatoeba export and record its corpus counts
2. **Train** the final v1 recipe (Tatoeba-only, focus-cap, 12 epochs)
3. **Evaluate** all checkpoints
4. **Inspect** sample translations
5. **Upload** to HuggingFace

Run cells top-to-bottom. You only need to set the Huggingface token in .env

## 1. Configuration

In [ ]:
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(".env")

from nllb_try.config import RunConfig
from nllb_try.corpus import GLOBAL_HOLDOUT_FRACTION, SPLIT_SEED, main_corpus
from nllb_try.downloadtatoeba import main_download

DATA_DIR = "data"

cfg = RunConfig(
    modelname="facebook/nllb-200-distilled-600M",
    source_langs_tatoeba=("nld", "gos", "deu", "eng", "spa"),
    source_langs_nllb=(
        "nld_Latn",
        "gos_Latn",
        "deu_Latn",
        "eng_Latn",
        "spa_Latn",
    ),
    new_lang_nllb="gos_Latn",
    similar_lang_nllb="nld_Latn",
    tatoeba_path=os.path.join(DATA_DIR, "tatoeba"),
    data_root_path=DATA_DIR,
    model_cache_path="hfacemodels",
    batch_size=256,
    num_epochs=12,
    warmup_steps=70,
    sampling_temperature=5.0,
    sampling_strategy="focus_cap",
    focus_lang_pair=("nld_Latn", "gos_Latn"),
    direction_strategy="random",
    parallel_data_paths=(),
    max_length=48,
    device="cuda:1"
)
print(f"Run ID: {cfg.run_id}")
print(f"Run dir: {cfg.run_dir}")

## 2. Download and Build Tatoeba Corpora

Downloads the latest mutable Tatoeba export and creates the global sentence-ID
split. The exact rows are not archived for this deprecated v1 model. Instead, a
small manifest records the download time, split settings, pair counts, and Git
revision alongside the training run.

In [ ]:
main_download(
    cfg.source_langs_tatoeba,
    redownload=True,
    tatoeba_path=cfg.tatoeba_path,
)
downloaded_at = datetime.now(timezone.utc).isoformat()

train_corpora = main_corpus(
    source_langs_tatoeba=cfg.source_langs_tatoeba,
    source_langs_nllb=cfg.source_langs_nllb,
    parallel_data_paths=(),
    cfg=cfg,
)

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
git_dirty = bool(subprocess.run(
    ["git", "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip())
corpus_counts = [
    {
        "source_lang_nllb": corpus.source_lang_nllb,
        "target_lang_nllb": corpus.target_lang_nllb,
        "total_rows": len(corpus.df_train) + len(corpus.df_validate),
        "train_rows": len(corpus.df_train),
        "validation_rows": len(corpus.df_validate),
    }
    for corpus in train_corpora
]
provenance = {
    "downloaded_at": downloaded_at,
    "source": "https://tatoeba.org/en/downloads",
    "snapshot_archived": False,
    "split_strategy": "global_sentence_id_holdout",
    "split_seed": SPLIT_SEED,
    "holdout_fraction": GLOBAL_HOLDOUT_FRACTION,
    "git_commit": git_commit,
    "git_worktree_dirty": git_dirty,
    "corpora": corpus_counts,
}
run_dir = Path(cfg.run_dir)
run_dir.mkdir(parents=True, exist_ok=True)
provenance_path = run_dir / "tatoeba_data_manifest.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"Saved dataset provenance to {provenance_path}")

print("\n--- Training corpora (focused multilingual) ---")
total = 0
for c in train_corpora:
    print(
        f"  {c.source_lang_nllb}-{c.target_lang_nllb}: "
        f"train={len(c.df_train):,}, val={len(c.df_validate):,}"
    )
    total += len(c.df_train)
print(f"  Total: {total:,} training pairs")

## 3. Train

In [ ]:
from nllb_try.train import main_train

main_train(train_corpora, cfg)
print(f"Training complete. Run dir: {cfg.run_dir}")

## 4. Evaluate

Evaluates all epoch checkpoints on this run's Tatoeba splits and the versioned
Kreuze validation split, including the untrained baseline for reference. Kreuze
is used only for OOD evaluation, never for v1 training. Do not assume that epoch
12 is best.

In [ ]:
from nllb_try.corpus import ParallelFileCorpus
from nllb_try.evaluate import main_evaluate

evaluation_corpora = [
    *train_corpora,
    ParallelFileCorpus("data/kreuze/kreuze_synthetic_gemma50.csv"),
]

main_evaluate(
    corpus_objects=evaluation_corpora,
    run_dir=cfg.run_dir,
    new_lang_nllb=cfg.new_lang_nllb,
    device=cfg.device,
    sample_size=750,
    include_baseline=True,
    include_train=False,
    verbose=True,
)
print("Evaluation complete — check the eval/ subfolder for metrics.csv and plots.")

## 5. Inspect Translations

Set `V1_BEST_EPOCH` only after reviewing the validation results. Keeping it at
`None` deliberately blocks inspection and upload.

In [ ]:
from nllb_try.evaluate import translate
from nllb_try.tokenizer_and_model_setup import setup_model_and_tokenizer

V1_BEST_EPOCH = 8  # Set this after reviewing validation metrics.
assert V1_BEST_EPOCH is not None, "Choose the best validation epoch first."
FINAL_MODEL_PATH = Path(cfg.run_dir) / "checkpoints" / f"epoch{V1_BEST_EPOCH}"
print(f"Loading {FINAL_MODEL_PATH}...")

model, tokenizer = setup_model_and_tokenizer(
    str(FINAL_MODEL_PATH),
    modelpath=cfg.model_cache_path,
    new_lang=cfg.new_lang_nllb,
    device=cfg.device,
)

# Sample sentences
sentences_nld = [
    "Ik ga morgen naar de stad.",
    "Het regent buiten. Neem een paraplu mee!",
    "De kat slaapt zachtjes op de bank.",
    "Wij drinken koffie bij oma.",
    "Hoe gaat het met jou?",
    "Die film was echt heel spannend.",
    "Het kind speelt in de tuin.",
    "Ik weet niet wat je bedoelt.",
    "De man die je gisteren zag, is mijn buurman.",
    "Het kind dat in de speeltuin speelt, is mijn neefje.",
]

print("\n--- Nederlands → Gronings ---")
translations_gos = []
for s in sentences_nld:
    t = translate(
        s, src_lang="nld_Latn", tgt_lang="gos_Latn", model=model, tokenizer=tokenizer
    )
    translations_gos.append(t)
    print(f"  NL:  {s}")
    print(f"  GOS: {t}\n")

print("--- Gronings → Nederlands (back-translation) ---")
for gos in translations_gos:
    back = translate(
        gos, src_lang="gos_Latn", tgt_lang="nld_Latn", model=model, tokenizer=tokenizer
    )
    print(f"  GOS: {gos}")
    print(f"  NL:  {back}\n")

In [ ]:
REPO_ID = "Tom9358/nllb-tatoeba-gos-nld-v1"

print(tokenizer.convert_tokens_to_ids("gos_Latn"))

print(translate(
    "Het tuinhuis van suikerruiker.",
    src_lang="nld_Latn",
    tgt_lang="gos_Latn",
    model=model,
    tokenizer=tokenizer,
))

## 6. Upload to HuggingFace

Log in and push the model + tokenizer. Set `HF_TOKEN` in `.env`; get a
write-enabled token from https://huggingface.co/settings/tokens.

In [ ]:
import os

import huggingface_hub
from dotenv import load_dotenv

load_dotenv(".env")
huggingface_hub.login(token=os.environ["HF_TOKEN"])

In [ ]:
REPO_ID = "Tom9358/nllb-tatoeba-gos-nld-v1"

assert FINAL_MODEL_PATH.is_dir(), FINAL_MODEL_PATH
upload_model, upload_tokenizer = setup_model_and_tokenizer(
    str(FINAL_MODEL_PATH),
    modelpath=cfg.model_cache_path,
    new_lang=cfg.new_lang_nllb,
    device=cfg.device,
)
upload_tokenizer.push_to_hub(
    REPO_ID,
    commit_message=f"Upload final v1 epoch {V1_BEST_EPOCH}",
)
upload_model.push_to_hub(
    REPO_ID,
    commit_message=f"Upload final v1 epoch {V1_BEST_EPOCH}",
)